In [1]:
# testing cell

sample_grid = []

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
import json
from datetime import datetime
import pandas as pd

def grid_parser(url):
# url = "https://www.cavalierdaily.com/article/2026/02/mini-crosswords-week-of-feb-16"

    options = Options()
    options.page_load_strategy = "eager"

    driver = webdriver.Chrome(options=options)
    driver.get(url)

    iframes = driver.find_elements(By.TAG_NAME, "iframe")

    rows = []

    for iframe in iframes:
        src = iframe.get_attribute("src")

        if not src or "crosshare" not in src:
            continue

        driver.switch_to.frame(iframe)

        script = driver.find_element(By.ID, "__NEXT_DATA__")
        json_text = script.get_attribute("innerHTML")

        data = json.loads(json_text)
        puzzle = data["props"]["pageProps"]["puzzle"]
        grid = puzzle["grid"]
        cols = puzzle["size"]["cols"]
        time = puzzle["publishTime"]
        dt = datetime.fromtimestamp(time/1000).date()
        
        words = []
        downs = [[] for _ in range(cols)]
        for i in range(0, len(grid), cols):
            across = (grid[i:i+5])
            word = "".join(across)
            word = word.strip(".")
            words.append(word)
            for j in range(cols):
                if grid[i+j] != ".":
                    downs[j].append(grid[i+j])
        
        for d in downs:
            words.append("".join(d))
        
        for w in words:
            rows.append({"Date": dt, "Word": w})

        driver.switch_to.default_content()

    df = pd.DataFrame(rows)

    driver.quit()

    return df

In [3]:
df = grid_parser(url = "https://www.cavalierdaily.com/article/2026/02/mini-crosswords-week-of-feb-16")

In [9]:
counts = df["Word"].value_counts()

df_filtered = df[df["Word"].isin(counts[counts > 2].index)]
print(df_filtered.head())

Empty DataFrame
Columns: [Date, Word]
Index: []
